## 1️⃣ Konfigürasyon ve Bağlantı Testi

In [ ]:
# Önce config modülünü import et
import sys
sys.path.insert(0, '..')  # Parent dizini ekle

from config import (
    get_s3_client, 
    validate_credentials, 
    print_config,
    AWS_REGION,
    S3_BUCKET
)

# Konfigürasyonu göster
print_config()

In [ ]:
# AWS bağlantısını test et
validate_credentials()

In [ ]:
# S3 client al
s3 = get_s3_client()

# Mevcut bucket'ları listele
response = s3.list_buckets()

print("📦 Mevcut Bucket'lar:")
print("=" * 40)
for bucket in response['Buckets']:
    print(f"  • {bucket['Name']}")
print(f"\nToplam: {len(response['Buckets'])} bucket")

---
## 2️⃣ Bucket Oluşturma (CREATE)

### Kurallar:
- Bucket adı **global unique** olmalı (tüm AWS'de tek)
- Sadece küçük harf, rakam ve tire (-) kullanılabilir
- 3-63 karakter arası
- Tire ile başlayamaz/bitemez

In [ ]:
# Test için yeni bucket adı
TEST_BUCKET = "kaan-test-bucket-12345"  # <- Unique bir isim seç!

def create_bucket(bucket_name: str, region: str = AWS_REGION):
    """
    Yeni S3 bucket oluştur.
    
    Args:
        bucket_name: Bucket adı (global unique olmalı)
        region: AWS region
    
    Returns:
        bool: Başarılı mı?
    """
    s3 = get_s3_client()
    
    try:
        # us-east-1 için LocationConstraint belirtme
        if region == "us-east-1":
            s3.create_bucket(Bucket=bucket_name)
        else:
            s3.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={"LocationConstraint": region}
            )
        print(f"✅ Bucket oluşturuldu: {bucket_name}")
        return True
    except s3.exceptions.BucketAlreadyExists:
        print(f"❌ Bu isim zaten kullanımda: {bucket_name}")
        return False
    except s3.exceptions.BucketAlreadyOwnedByYou:
        print(f"ℹ️  Bu bucket zaten sizin: {bucket_name}")
        return True
    except Exception as e:
        print(f"❌ Hata: {e}")
        return False

# Bucket oluştur
# create_bucket(TEST_BUCKET)  # <- Uncomment et ve çalıştır!

---
## 3️⃣ Dosya Yükleme (PUT/UPLOAD)

3 yöntem var:
1. `put_object()` - Küçük dosyalar için (memory'de)
2. `upload_file()` - Disk'ten yükleme
3. `upload_fileobj()` - File-like object'ten yükleme

In [ ]:
def upload_text_to_s3(bucket: str, key: str, content: str):
    """
    Metin içeriği S3'e yükle.
    
    Args:
        bucket: Bucket adı
        key: S3'teki dosya yolu (örn: 'folder/file.txt')
        content: Yüklenecek metin
    """
    s3 = get_s3_client()
    
    s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=content.encode('utf-8'),
        ContentType='text/plain'
    )
    print(f"✅ Yüklendi: s3://{bucket}/{key}")

def upload_json_to_s3(bucket: str, key: str, data: dict):
    """
    JSON veri S3'e yükle.
    """
    import json
    s3 = get_s3_client()
    
    s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=json.dumps(data, ensure_ascii=False).encode('utf-8'),
        ContentType='application/json'
    )
    print(f"✅ JSON yüklendi: s3://{bucket}/{key}")

def upload_file_to_s3(bucket: str, key: str, local_path: str):
    """
    Lokal dosyayı S3'e yükle.
    """
    s3 = get_s3_client()
    
    s3.upload_file(local_path, bucket, key)
    print(f"✅ Dosya yüklendi: {local_path} → s3://{bucket}/{key}")

# Örnek kullanım:
# upload_text_to_s3(S3_BUCKET, "test/hello.txt", "Merhaba Dünya!")
# upload_json_to_s3(S3_BUCKET, "test/data.json", {"name": "Kaan", "city": "Istanbul"})

---
## 4️⃣ Dosya Listeleme (LIST)

In [ ]:
def list_objects(bucket: str, prefix: str = "", max_keys: int = 100):
    """
    Bucket içindeki dosyaları listele.
    
    Args:
        bucket: Bucket adı
        prefix: Filtreleme için prefix (örn: 'tiles/')
        max_keys: Maksimum sonuç sayısı
    
    Returns:
        list: Dosya bilgileri
    """
    s3 = get_s3_client()
    
    response = s3.list_objects_v2(
        Bucket=bucket,
        Prefix=prefix,
        MaxKeys=max_keys
    )
    
    objects = response.get('Contents', [])
    
    print(f"📁 s3://{bucket}/{prefix}")
    print("=" * 60)
    
    total_size = 0
    for obj in objects:
        size_kb = obj['Size'] / 1024
        total_size += obj['Size']
        print(f"  {obj['Key']:<45} {size_kb:>10.1f} KB")
    
    print("=" * 60)
    print(f"Toplam: {len(objects)} dosya, {total_size/(1024*1024):.2f} MB")
    
    return objects

# Mevcut bucket'ı listele
# list_objects(S3_BUCKET, prefix="tiles/", max_keys=10)

---
## 5️⃣ Dosya İndirme (GET/DOWNLOAD)

In [ ]:
def get_object_content(bucket: str, key: str) -> bytes:
    """
    S3'ten dosya içeriğini oku.
    
    Args:
        bucket: Bucket adı
        key: Dosya yolu
    
    Returns:
        bytes: Dosya içeriği
    """
    s3 = get_s3_client()
    
    response = s3.get_object(Bucket=bucket, Key=key)
    content = response['Body'].read()
    
    print(f"✅ İndirildi: s3://{bucket}/{key}")
    print(f"   Boyut: {len(content)/1024:.1f} KB")
    print(f"   Content-Type: {response.get('ContentType', 'N/A')}")
    
    return content

def download_file(bucket: str, key: str, local_path: str):
    """
    S3'ten dosyayı lokale indir.
    """
    s3 = get_s3_client()
    
    s3.download_file(bucket, key, local_path)
    print(f"✅ İndirildi: s3://{bucket}/{key} → {local_path}")

# Örnek:
# content = get_object_content(S3_BUCKET, "index/quadkey_index.csv")
# print(content[:500].decode('utf-8'))  # İlk 500 karakter

---
## 6️⃣ Dosya Silme (DELETE)

In [ ]:
def delete_object(bucket: str, key: str):
    """
    S3'ten tek dosya sil.
    
    ⚠️ DİKKAT: Silinen dosya geri getirilemez!
    """
    s3 = get_s3_client()
    
    s3.delete_object(Bucket=bucket, Key=key)
    print(f"🗑️  Silindi: s3://{bucket}/{key}")

def delete_multiple_objects(bucket: str, keys: list):
    """
    Birden fazla dosya sil (batch delete).
    
    Args:
        bucket: Bucket adı
        keys: Silinecek dosya yolları listesi
    """
    s3 = get_s3_client()
    
    # S3 batch delete formatı
    delete_objects = {'Objects': [{'Key': key} for key in keys]}
    
    response = s3.delete_objects(Bucket=bucket, Delete=delete_objects)
    
    deleted = response.get('Deleted', [])
    errors = response.get('Errors', [])
    
    print(f"🗑️  Silindi: {len(deleted)} dosya")
    if errors:
        print(f"❌ Hata: {len(errors)} dosya silinemedi")

# Örnek:
# delete_object(S3_BUCKET, "test/hello.txt")

---
## 7️⃣ Bucket Silme

⚠️ **Kural:** Bucket silinmeden önce içi boşaltılmalı!

In [ ]:
def delete_bucket(bucket: str, force: bool = False):
    """
    Bucket sil.
    
    Args:
        bucket: Bucket adı
        force: True ise önce içindekileri sil
    
    ⚠️ DİKKAT: Geri alınamaz!
    """
    s3 = get_s3_client()
    
    if force:
        # Önce tüm dosyaları sil
        print(f"🗑️  Bucket içeriği siliniyor: {bucket}")
        
        paginator = s3.get_paginator('list_objects_v2')
        for page in paginator.paginate(Bucket=bucket):
            objects = page.get('Contents', [])
            if objects:
                keys = [obj['Key'] for obj in objects]
                delete_multiple_objects(bucket, keys)
    
    try:
        s3.delete_bucket(Bucket=bucket)
        print(f"✅ Bucket silindi: {bucket}")
    except Exception as e:
        print(f"❌ Bucket silinemedi: {e}")
        print("   Bucket boş olmayabilir. force=True deneyin.")

# Örnek:
# delete_bucket(TEST_BUCKET)  # Boş bucket
# delete_bucket(TEST_BUCKET, force=True)  # İçi dolu bucket

---
## 📝 Alıştırma: Kendin Dene!

Aşağıdaki hücreleri doldurup çalıştır:

In [ ]:
# ALIŞTIRMA 1: Test bucket'ı oluştur
# MY_BUCKET = "..."  # Unique bir isim seç
# create_bucket(MY_BUCKET)

In [ ]:
# ALIŞTIRMA 2: Bir JSON dosyası yükle
# upload_json_to_s3(MY_BUCKET, "test/my_data.json", {"hello": "world"})

In [ ]:
# ALIŞTIRMA 3: Yüklediğin dosyayı oku
# content = get_object_content(MY_BUCKET, "test/my_data.json")
# print(content.decode('utf-8'))

In [ ]:
# ALIŞTIRMA 4: Dosyayı sil
# delete_object(MY_BUCKET, "test/my_data.json")

In [ ]:
# ALIŞTIRMA 5: Bucket'ı sil
# delete_bucket(MY_BUCKET)

---
## ✅ Özet

| İşlem | Fonksiyon | boto3 Method |
|-------|-----------|---------------|
| Bucket oluştur | `create_bucket()` | `s3.create_bucket()` |
| Dosya yükle | `upload_*()` | `s3.put_object()`, `s3.upload_file()` |
| Dosya listele | `list_objects()` | `s3.list_objects_v2()` |
| Dosya indir | `get_object_content()` | `s3.get_object()` |
| Dosya sil | `delete_object()` | `s3.delete_object()` |
| Bucket sil | `delete_bucket()` | `s3.delete_bucket()` |